# Web Analytics: PageRank Algorithm on Wikipedia Link Graph

## Project Overview
This project implements the **PageRank algorithm from scratch** using power iteration on a large-scale real-world web graph: the **Wikipedia Link Graph Dataset (100,000 Pages)**.

### Objectives:
1. **Data Loading and Cleaning**: Inspect and clean the raw Wikipedia graph, eliminate duplicate edges and self-loops, and map nodes to contiguous zero-indexed integers.
2. **Graph Construction**: Assemble a highly efficient column-stochastic sparse transition matrix in **scipy CSR format** ($M$).
3. **PageRank from Scratch**: Implement the classic PageRank power iteration with uniform redistribution for dangling nodes ($d = 0.85$, $\epsilon = 10^{-6}$, max 100 iterations).
4. **Validation against NetworkX**: Benchmark accuracy (maximum absolute difference and top-20 overlap) and computational runtime against `networkx.pagerank`.
5. **In-Depth Analysis & Visualizations**:
   - Top-20 pages table with in/out-degrees.
   - Power iteration convergence curve (L1 error vs iteration, log scale).
   - Distribution of PageRank scores (log-scaled histogram).
   - Correlation analysis between PageRank and In-Degree (log-log scatter plot).
   - Sensitivity analysis across damping factors $d \in [0.5, 0.7, 0.85, 0.95, 0.99]$.
   - Subgraph visualization of high-ranking pages.


---
## Step 0: Dataset Inspection

Before executing graph algorithms, we list the files in `./data/`, inspect their schema, dimensions, and sample rows.


In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np

# Configure standard output encoding safely if supported
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# List files and file sizes in ./data/
print("=== Files in ./data/ ===")
data_files = os.listdir('data')
for f in sorted(data_files):
    p = os.path.join('data', f)
    size_mb = os.path.getsize(p) / (1024 * 1024)
    print(f" - {f:<25} ({size_mb:>8.2f} MB)")

# Inspect pages_export.csv
pages_sample = pd.read_csv('data/pages_export.csv', nrows=10)
print("\n=== pages_export.csv (Sample Preview) ===")
print(f"Sample Shape: {pages_sample.shape}")
print("Columns:", pages_sample.columns.tolist())
print("Data Types:\n", pages_sample.dtypes)
display(pages_sample[['id', 'title', 'language', 'content_length', 'word_count', 'url']])

# Inspect links_export.csv
links_sample = pd.read_csv('data/links_export.csv', nrows=10)
print("\n=== links_export.csv (Sample Preview) ===")
print(f"Sample Shape: {links_sample.shape}")
print("Columns:", links_sample.columns.tolist())
print("Data Types:\n", links_sample.dtypes)
display(links_sample[['id', 'source_title', 'target_title', 'position', 'depth', 'source_url', 'target_url']])


=== Files in ./data/ ===
 - DATASET_DESCRIPTION.md    (    0.01 MB)
 - README.md                 (    0.00 MB)
 - WHATSAPP_SHARE.txt        (    0.00 MB)
 - clean_edges.npz           (   18.03 MB)
 - graph.json                ( 2717.78 MB)
 - links_export.csv          ( 4886.73 MB)
 - pages_export.csv          (   71.54 MB)

=== pages_export.csv (Sample Preview) ===
Sample Shape: (10, 9)
Columns: ['id', 'title', 'language', 'content_length', 'word_count', 'categories', 'infobox', 'created_at', 'url']
Data Types:
 id                  int64
title                 str
language              str
content_length      int64
word_count          int64
categories            str
infobox           float64
created_at            str
url                   str
dtype: object


,id,title,language,content_length,word_count,url
0,1,The Loner (TV series),en,11901,1804,https://en.wikipedia.org/wiki/The_Loner_(TV_se...
1,2,She's a Hottie,en,5662,923,https://en.wikipedia.org/wiki/She's_a_Hottie
2,3,Point of No Return (TV series),en,5928,992,https://en.wikipedia.org/wiki/Point_of_No_Retu...
3,610,Underwater exploration,en,70564,9767,https://en.wikipedia.org/wiki/Underwater_explo...
4,612,Adati,en,15214,2141,https://en.wikipedia.org/wiki/Adati
5,613,Henry Draper Catalog,en,11743,1821,https://en.wikipedia.org/wiki/Henry_Draper_Cat...
6,616,The World Is Not Enough,en,39596,6391,https://en.wikipedia.org/wiki/The_World_Is_Not...
7,618,Bolesław II the Generous,en,15965,2450,https://en.wikipedia.org/wiki/Bolesław_II_the_...
8,620,Ismat Gayibov,en,1763,256,https://en.wikipedia.org/wiki/Ismat_Gayibov
9,621,Deep-focus earthquake,en,16069,2350,https://en.wikipedia.org/wiki/Deep-focus_earth...



=== links_export.csv (Sample Preview) ===
Sample Shape: (10, 9)
Columns: ['id', 'source_title', 'target_title', 'language', 'position', 'depth', 'created_at', 'source_url', 'target_url']
Data Types:
 id              int64
source_title      str
target_title      str
language          str
position        int64
depth           int64
created_at        str
source_url        str
target_url        str
dtype: object


,id,source_title,target_title,position,depth,source_url,target_url
0,1,The Loner (TV series),20th Century-Fox Television,0,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/20th_Century-Fox...
1,2,The Loner (TV series),Al Checco,1,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Al_Checco
2,3,The Loner (TV series),Alan Baxter (actor),2,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Alan_Baxter_(actor)
3,4,The Loner (TV series),Alexander Courage,3,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Alexander_Courage
4,5,The Loner (TV series),Allan Sherman,4,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Allan_Sherman
5,6,The Loner (TV series),Allen H. Miner,5,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Allen_H._Miner
6,7,The Loner (TV series),American Civil War,6,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/American_Civil_War
7,8,The Loner (TV series),American west,7,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/American_west
8,9,The Loner (TV series),Anne Baxter,8,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Anne_Baxter
9,10,The Loner (TV series),Barry Sullivan (actor),9,1,https://en.wikipedia.org/wiki/The_Loner_(TV_se...,https://en.wikipedia.org/wiki/Barry_Sullivan_(...


### Inspection Summary:
1. **`pages_export.csv`**: Contains complete metadata for the 100,000 Wikipedia articles crawled (unique `id`, `title`, word counts, and Wikipedia URLs).
2. **`links_export.csv`**: Represents directed hyperlinks from `source_title` to `target_title` (28.85 million raw links).
3. **Graph Boundaries**: Because the crawler crawled outward up to depth 5, raw links include edges pointing outside the 100,000 crawled set. We filter to retain internal edges within the 100,000-page corpus, treating nodes with no outgoing internal links as dangling nodes.


---
## Step 1: Data Loading, Cleaning, and Graph Statistics

In this section:
1. We load the 100,000 Wikipedia articles and map each page title and ID to a unique contiguous integer index $[0, N-1]$.
2. We clean the directed edge list by:
   - Filtering for links within the 100,000 catalog.
   - Eliminating all **self-loops** ($src == dst$).
   - Removing all **duplicate directed edges**.
3. We compute and report fundamental graph properties:
   - Total nodes ($N$) and clean directed edges ($E$).
   - Average out-degree.
   - Maximum in-degree and the corresponding page title.
   - Count and percentage of **dangling nodes** (out-degree $= 0$).
   - Directed graph density: $\text{Density} = \frac{E}{N(N-1)}$.


In [2]:
def load_and_clean_graph(data_dir='data', cache_file='data/clean_edges.npz'):
    """
    Loads Wikipedia pages, builds integer index mappings, cleans the edge list,
    removes duplicate edges and self-loops, and caches clean edges for rapid loading.
    """
    t_start = time.time()
    
    # 1. Load pages catalog
    pages_path = os.path.join(data_dir, 'pages_export.csv')
    print(f"Loading page catalog from {pages_path}...")
    pages_df = pd.read_csv(pages_path, usecols=['id', 'title'])
    pages_df['title'] = pages_df['title'].astype(str)
    N = len(pages_df)
    
    # Build bidirectional mappings
    title_to_idx = {title: i for i, title in enumerate(pages_df['title'])}
    idx_to_title = {i: title for i, title in enumerate(pages_df['title'])}
    idx_to_page_id = {i: pid for i, pid in enumerate(pages_df['id'])}
    
    print(f"Successfully loaded {N:,} nodes into catalog.")
    
    # 2. Check if cached clean edges exist
    if os.path.exists(cache_file):
        print(f"Loading cached clean edges from {cache_file}...")
        data = np.load(cache_file)
        src = data['src']
        dst = data['dst']
        edges_df = pd.DataFrame({'src': src, 'dst': dst})
    else:
        print("Parsing and cleaning raw links in chunks...")
        links_path = os.path.join(data_dir, 'links_export.csv')
        edge_chunks = []
        chunk_size = 1000000
        
        for chunk in pd.read_csv(links_path, chunksize=chunk_size, usecols=['source_title', 'target_title']):
            chunk['target_title'] = chunk['target_title'].astype(str)
            mask = chunk['target_title'].isin(title_to_idx)
            valid = chunk[mask]
            
            s_idx = [title_to_idx[s] for s in valid['source_title']]
            d_idx = [title_to_idx[t] for t in valid['target_title']]
            
            cdf = pd.DataFrame({'src': s_idx, 'dst': d_idx}, dtype=np.int32)
            # Remove self-loops
            cdf = cdf[cdf['src'] != cdf['dst']]
            edge_chunks.append(cdf)
            
        edges_df = pd.concat(edge_chunks, ignore_index=True)
        # Remove duplicate edges
        edges_df = edges_df.drop_duplicates(subset=['src', 'dst']).reset_index(drop=True)
        # Cache for fast subsequent runs
        np.savez_compressed(cache_file, src=edges_df['src'].values, dst=edges_df['dst'].values)
        print(f"Saved clean edges to {cache_file}.")
        
    print(f"Clean edge list prepared in {time.time() - t_start:.2f} seconds.")
    return N, edges_df, title_to_idx, idx_to_title, idx_to_page_id

N, edges_df, title_to_idx, idx_to_title, idx_to_page_id = load_and_clean_graph()
print(f"Clean edges DataFrame shape: {edges_df.shape}")
display(edges_df.head(10))


Loading page catalog from data\pages_export.csv...


Successfully loaded 100,000 nodes into catalog.
Loading cached clean edges from data/clean_edges.npz...
Clean edge list prepared in 0.79 seconds.
Clean edges DataFrame shape: (8572718, 2)


,src,dst
0,0,19619
1,0,13114
2,0,43374
3,0,7077
4,0,41548
5,0,93835
6,1,44653
7,1,21079
8,1,40579
9,1,28968


In [3]:
# Verification of data cleaning
print("=== Verification of Edge List Cleaning ===")
num_self_loops = (edges_df['src'] == edges_df['dst']).sum()
num_duplicates = edges_df.duplicated(subset=['src', 'dst']).sum()
print(f"Self-loops count: {num_self_loops} (Expected: 0)")
print(f"Duplicate edges count: {num_duplicates} (Expected: 0)")
assert num_self_loops == 0, "Error: Self-loops detected in clean edge list!"
assert num_duplicates == 0, "Error: Duplicate edges detected in clean edge list!"
print("Data integrity check passed: 100% clean directed edge list.")


=== Verification of Edge List Cleaning ===


Self-loops count: 0 (Expected: 0)
Duplicate edges count: 0 (Expected: 0)
Data integrity check passed: 100% clean directed edge list.


In [4]:
# Compute and display graph statistics
def print_graph_statistics(N, edges_df, idx_to_title):
    E = len(edges_df)
    
    # Degree computations using fast numpy bincount
    out_degrees = np.bincount(edges_df['src'], minlength=N)
    in_degrees = np.bincount(edges_df['dst'], minlength=N)
    
    avg_out_degree = E / N
    max_in_degree = int(in_degrees.max())
    max_in_idx = int(in_degrees.argmax())
    max_in_title = idx_to_title[max_in_idx]
    
    dangling_mask = (out_degrees == 0)
    num_dangling = int(np.sum(dangling_mask))
    pct_dangling = (num_dangling / N) * 100
    
    # Graph density: E / (N * (N - 1))
    density = E / (N * (N - 1))
    
    print("=" * 60)
    print("                 WIKIPEDIA GRAPH STATISTICS")
    print("=" * 60)
    print(f"Total Nodes (N)               : {N:,}")
    print(f"Total Directed Edges (E)      : {E:,}")
    print(f"Average Out-Degree            : {avg_out_degree:.2f}")
    print(f"Maximum In-Degree             : {max_in_degree:,} (Page: '{max_in_title}')")
    print(f"Dangling Nodes (out-degree 0) : {num_dangling:,} ({pct_dangling:.2f}%)")
    print(f"Graph Density                 : {density:.6e}")
    print("=" * 60)
    
    return {
        'N': N,
        'E': E,
        'avg_out_degree': avg_out_degree,
        'max_in_degree': max_in_degree,
        'max_in_title': max_in_title,
        'num_dangling': num_dangling,
        'pct_dangling': pct_dangling,
        'density': density,
        'out_degrees': out_degrees,
        'in_degrees': in_degrees,
        'dangling_mask': dangling_mask
    }

graph_stats = print_graph_statistics(N, edges_df, idx_to_title)


                 WIKIPEDIA GRAPH STATISTICS
Total Nodes (N)               : 100,000
Total Directed Edges (E)      : 8,572,718
Average Out-Degree            : 85.73
Maximum In-Degree             : 51,534 (Page: 'ISBN (identifier)')
Dangling Nodes (out-degree 0) : 205 (0.21%)
Graph Density                 : 8.572804e-04


### Interpretation of Graph Statistics:
- **Graph Scale**: With **100,000 nodes** and **8,572,718 directed edges**, the network represents an extensive, rich hyperlinked knowledge graph.
- **Sparsity**: The density is approximately $8.57 \times 10^{-4}$ (0.0857%), meaning that only a small fraction of all possible $N(N-1) \approx 10^{10}$ directed connections exist. Storing this graph as a dense $100,000 \times 100,000$ matrix would require $10^{10} \times 8$ bytes $\approx 80\text{ GB}$ of RAM, completely exhausting memory. This empirically demonstrates why a **sparse matrix (scipy CSR format)** is strictly required.
- **Dangling Nodes**: There are **205 dangling nodes** (0.21% of the graph) that possess an out-degree of 0 within the corpus. In standard random walk models without teleportation, probability mass accumulating at dangling nodes would leak out of the system, causing total rank to diminish to 0. Our PageRank implementation will explicitly handle these 205 nodes by redistributing their probability mass uniformly across all $N$ pages at each iteration.
- **Hubs & Authorities**: The maximum in-degree page is **'ISBN (identifier)'** with **51,534 incoming links**, reflecting Wikipedia's citation and bibliographic structure where reference sections frequently link to standard identifier definitions.
